# Cross-dimension sensitivity (d = 10 / 20 / 50)

Does the choice of d change the analysis? One paper-style figure for our trajectories: a panel per
d (one representative medoid path per cluster, clusters Hungarian-matched across d so a colour
means the same fate in every panel) plus two metrics -- **trajectory agreement** (how far apart
the SAME cell's two trajectories are in the coordinates the two dimensions share, as % of cloud
radius; lower = better) and **cluster agreement** (fraction of shared cells landing in matched
clusters; higher = better).

The comparison is meaningful because the PC bases are exactly nested and the row order is identical
across d (verified bit-for-bit), so a row index means the same cell and the leading shared
coordinates are literally the same axes. d=20 uses the settled tuned seed (embryoid 5, statefate
2); other dims use their own `d<k>_tune/seed<k>/` when present, else the flat `d<k>/` run.

In [ ]:
import os, sys, json, glob, itertools
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from scipy.stats import pearsonr
from scipy.optimize import linear_sum_assignment

_here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
# this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
for _d in (_here, os.path.abspath(os.path.join(_here, os.pardir, os.pardir, "tools"))):
    if _d not in sys.path:
        sys.path.insert(0, _d)
import _analysis_common as A
import _repo as P

## >>> PARAMETERS <<< + load OUR trajectories/clusters/DMs per available d

In [ ]:
DATASET = globals().get("DATASET", "embryoid")
DIMS    = [10, 20, 50]
PRIMARY = "oursUOTmaps"   # our method under study
REFDIM  = 20                                           # cluster colors matched to this dim's labels
COMMON_D = 10  # coordinates the metric is measured in

# The settled trajectory seeds. Fixed here rather than passed in, because the whole analysis
# (clusters, fate genes, the trajectory figure) is reported at these seeds -- a cross-dimension claim
# computed at a different one would not be about the analysis the paper shows.
SEED_BY_DS = {"embryoid": "5", "statefate": "2"}
TUNE_SEED  = globals().get("TUNE_SEED", SEED_BY_DS.get(DATASET, ""))

SHIPPED = globals().get("SHIPPED", 1)   # 1 = read the shipped paper results; 0 = your new_results/ run

FS = globals().get("FS", 1.30)        # same text scale as `figures`
plt.rcParams.update({"font.size": 11 * FS, "axes.titlesize": 12 * FS, "axes.labelsize": 11 * FS,
                     "xtick.labelsize": 10 * FS, "ytick.labelsize": 10 * FS,
                     "legend.fontsize": 9 * FS, "figure.titlesize": 13 * FS})
cmap = plt.get_cmap("tab10")


def run_dir(d, seed):
    """Folder for one (dimension, seed) run. `seed=None` = the flat run."""
    flat = A.results_dir(DATASET, d, shipped=SHIPPED)
    if seed in (None, "", "flat"):
        return flat
    return os.path.join(os.path.dirname(flat), f"d{d}_tune", f"seed{seed}")


def available_seeds(d):
    """Seeds with a saved clustering at this dimension, newest convention first."""
    base = os.path.join(os.path.dirname(A.results_dir(DATASET, d, shipped=SHIPPED)), f"d{d}_tune")
    out = []
    for pth in sorted(glob.glob(os.path.join(base, "seed*"))):
        if os.path.exists(os.path.join(pth, f"clusters_{DATASET}{d}_{PRIMARY}.npy")):
            out.append(os.path.basename(pth)[len("seed"):])
    return out


def load_run(d, seed):
    """One run as a dict, or None. `idx` = the GLOBAL day-0 row ids its arrays refer to.

    `startidx_*.npy` is authoritative when present (statefate's runs start from the published 3000,
    whose indices are SCATTERED through the 28249 -- 14396, 16752, ... -- so assuming a prefix
    silently compares unrelated cells). Without it the convention is `trajectory_analysis.py`'s:
    the first `N_CLUST_CELLS` rows."""
    src = run_dir(d, seed)
    tf = os.path.join(src, f"traj_{DATASET}{d}_{PRIMARY}.npy")
    cf = os.path.join(src, f"clusters_{DATASET}{d}_{PRIMARY}.npy")
    if not (os.path.exists(tf) and os.path.exists(cf)):
        return None
    lab = np.load(cf)
    sif = os.path.join(src, f"startidx_{DATASET}{d}_{PRIMARY}.npy")
    idx = np.load(sif)[:len(lab)].astype(int) if os.path.exists(sif) else np.arange(len(lab), int)
    dmf = os.path.join(src, f"dm_{DATASET}{d}_{PRIMARY}.npy")
    mjf = os.path.join(src, f"meta_{DATASET}{d}_{PRIMARY}.json")
    meta = json.load(open(mjf)) if os.path.exists(mjf) else {}
    traj = np.load(tf)
    # the cached DM is deterministic from the trajectory (`pairwise_aligned_euclid`), so when the
    # cache is absent (the release ships trajectories, not the ~70 MB-per-seed DM files) it is
    # recomputed here -- a few seconds per run -- instead of degrading the DM-correlation column.
    dm = np.load(dmf).astype(float) if os.path.exists(dmf) else A.pairwise_aligned_euclid(traj)
    return dict(dim=d, seed=seed, src=src, traj=traj, clusters=lab, idx=idx, dm=dm,
                fidelity=meta.get("metrics", {}).get("fidelity", float("nan")))


def _rows(run, cells):
    """Positions, within THIS run's arrays, of the given global cell ids.

    Never index a run's arrays with a global id directly: a tuned statefate run holds 3000 rows in
    the published order, so a global id reads a different cell entirely."""
    pos = {int(g): i for i, g in enumerate(run["idx"])}
    return np.array([pos[int(c)] for c in cells], dtype=int)


def common_cells(runs):
    return np.array(sorted(set.intersection(*[set(r["idx"].tolist()) for r in runs])))


def match_labels(l_ref, l_other):
    """Hungarian-relabel `l_other`'s cluster ids onto `l_ref`'s; returns (relabeled, agreement)."""
    K = int(max(l_ref.max(), l_other.max())) + 1
    Cm = confusion_matrix(l_ref, l_other, labels=np.arange(K))
    r, c = linear_sum_assignment(-Cm)
    mapping = {int(c_): int(r_) for r_, c_ in zip(r, c)}
    bm = np.array([mapping.get(int(x), int(x)) for x in l_other])
    return bm, float((l_ref == bm).mean())


def path_deviation(rA, rB, cells, k=COMMON_D, permute=False, seed=0):
    """(% of cloud radius, raw) between the SAME cells' trajectories in two runs.

    Measured in the leading `k` coordinates -- the same axes in both runs, because the PC bases are
    nested. Per cell it is the mean over time of the distance between its two paths; the normaliser
    is the mean over time of the RMS distance of cells from their centroid, so the number reads as
    "the two runs place the same cell this far apart, relative to how big the cloud is".

    `permute=True` shuffles B's cell correspondence, which is the NO-CORRESPONDENCE reference: it is
    what this metric reads when the two runs share nothing but their marginal distribution."""
    T = min(rA["traj"].shape[0], rB["traj"].shape[0])
    a = np.asarray(rA["traj"][:T][:, _rows(rA, cells), :k], float)
    b = np.asarray(rB["traj"][:T][:, _rows(rB, cells), :k], float)
    if permute:
        b = b[:, np.random.default_rng(seed).permutation(b.shape[1]), :]
    dev = np.linalg.norm(a - b, axis=-1).mean()
    rad = np.mean([np.sqrt(((x - x.mean(0)) ** 2).sum(1).mean())
                   for x in (a.reshape(-1, k), b.reshape(-1, k))])
    return (100.0 * dev / rad if rad else np.nan), float(dev)


def pair_metrics(rA, rB, cells=None):
    """The two headline metrics (+ ARI and the DM correlation) between any two runs."""
    cells = common_cells([rA, rB]) if cells is None else cells
    ra, rb = _rows(rA, cells), _rows(rB, cells)
    la, lb = rA["clusters"][ra], rB["clusters"][rb]
    _, agree = match_labels(la, lb)
    pct, raw = path_deviation(rA, rB, cells)
    shuf, _ = path_deviation(rA, rB, cells, permute=True)
    # chance level for a HUNGARIAN-MATCHED agreement is well above 1/K, because the matching picks
    # the best of K! permutations; estimate it by scoring a shuffled label vector the same way
    rng = np.random.default_rng(0)
    chance = float(np.mean([match_labels(la, rng.permutation(lb))[1] for _ in range(5)]))
    r_dm = np.nan
    if rA["dm"] is not None and rB["dm"] is not None:
        iu = np.triu_indices(len(cells), 1)
        r_dm = float(pearsonr(rA["dm"][np.ix_(ra, ra)][iu], rB["dm"][np.ix_(rb, rb)][iu])[0])
    return {"path_dev_pct": pct, "path_dev_raw": raw, "path_dev_shuffled": shuf,
            "cluster_agree": 100.0 * agree, "cluster_agree_chance": 100.0 * chance,
            "ARI": float(adjusted_rand_score(la, lb)), "dm_pearson": r_dm,
            "n_cells": int(len(cells))}


# ---- which run represents each dimension --------------------------------------------------------
# DEFAULT: the settled seed at EVERY dimension. Holding the seed fixed is what makes the comparison
# about the dimension; if d=10 used seed 3 and d=20 seed 5, a disagreement could be either.
SEED_BY_DIM = {d: TUNE_SEED for d in DIMS}    # per-dimension override: edit this dict

D = {}
for d in DIMS:
    r = load_run(d, SEED_BY_DIM[d]) or load_run(d, None)
    if r is None:
        print(f"  d={d}: MISSING (no seed{SEED_BY_DIM[d]} and no flat run)")
        continue
    D[d] = r
    print(f"  d={d:2d} [seed {r['seed'] or 'flat'}] traj {r['traj'].shape} | "
          f"clusters {np.bincount(r['clusters']).tolist()} on {len(r['clusters'])} cells | "
          f"own W2 {r['fidelity']:.3f}")
AVAIL = sorted(D)
assert AVAIL, f"no {PRIMARY} outputs found for any d -- run the tune file at each dimension first"
REFDIM = REFDIM if REFDIM in AVAIL else AVAIL[-1]

COMMON = common_cells([D[d] for d in AVAIL])
print(f"\n{DATASET} | dims={AVAIL} | reference dim = {REFDIM} | seeds "
      f"{ {d: D[d]['seed'] for d in AVAIL} }")
print(f"  cells clustered in EVERY dimension: {len(COMMON)}"
      + ("" if len(COMMON) >= 500 else "   ** small -- the per-cell metrics are thin **"))

## Cluster matching + the pairwise metrics
Everything is indexed through `_rows`, never by position: two runs' arrays are only comparable
cell-by-cell after mapping both onto the same global row ids.

In [ ]:
# relabel every dim's clusters onto REFDIM's ids, so a colour is the same fate in every panel
for d in AVAIL:
    if d == REFDIM:
        D[d]["clusters_m"], D[d]["overlap_ref"] = D[d]["clusters"], 1.0
        continue
    sub_ref = D[REFDIM]["clusters"][_rows(D[REFDIM], COMMON)]
    sub_oth = D[d]["clusters"][_rows(D[d], COMMON)]
    _, ov = match_labels(sub_ref, sub_oth)
    K = int(max(sub_ref.max(), sub_oth.max())) + 1
    r, c = linear_sum_assignment(-confusion_matrix(sub_ref, sub_oth, labels=np.arange(K)))
    m = {int(c_): int(r_) for r_, c_ in zip(r, c)}
    # mapping estimated on the common cells, APPLIED to all of this dim's cells so the panel can
    # still draw every cluster it has
    D[d]["clusters_m"] = np.array([m.get(int(x), int(x)) for x in D[d]["clusters"]])
    D[d]["overlap_ref"] = ov

pairs = {}
for d1, d2 in itertools.combinations(AVAIL, 2):
    m = pair_metrics(D[d1], D[d2], COMMON)
    m["seeds"] = [D[d1]["seed"], D[d2]["seed"]]
    pairs[f"d{d1}-d{d2}"] = m

print(f"\nOurs across d  (n = {len(COMMON)} cells, leading {COMMON_D} PCs)")
print(f"  {'pair':10s}{'path dev %':>12}{'cluster agree %':>17}{'ARI':>7}{'DM r':>7}")
for k, v in pairs.items():
    print(f"  {k:10s}{v['path_dev_pct']:12.1f}"
          f"{v['cluster_agree']:17.1f}{v['ARI']:7.2f}{v['dm_pearson']:7.2f}")

## THE figure -- one panel per d, plus the legend + metrics column
Same construction as `figures.fig_compare`: a gridspec whose last column is a fixed number of
INCHES wide (it holds text and bars, which do not shrink when panels are added), `constrained_layout`
because the column is a nested gridspec, and the legends excluded from the layout solve.

In [ ]:
# Wider than the trajectory figure's 3.30: the metric names here ("trajectory deviation (%)") are longer than
# "mean W2", and the axis label carries the direction phrase after them. At 3.30 both x-labels were
# cut off mid-word.
RIGHT_W    = 4.75   # widened again at FS 1.30: the axis labels were clipping
RIGHT_ROWS = (0.40, 0.30, 0.30)      # legend | trajectory agreement | cluster agreement
RIGHT_HSPACE = 0.26


def draw_dim_panel(ax, d, n_bg=None):
    """Light data background (PC1/2) + ONE representative (medoid) path per cluster.

    Medoid, not mean: the mean of several curved paths is a curve no cell actually took, while the
    medoid (smallest total DM distance to the rest of its cluster) IS one of the cells. Where no DM
    was saved the first member stands in, which is arbitrary but honest -- it is still a real path."""
    tr, lab, DM = D[d]["traj"], D[d]["clusters_m"], D[d]["dm"]
    for X in A.load_data(DATASET, d)[0]:
        P = np.asarray(X)[:, :2]
        step = max(1, len(P) // (n_bg or 700))
        ax.scatter(P[::step, 0], P[::step, 1], s=2.5, color="0.88", alpha=0.5, zorder=1,
                   rasterized=True)
    for c in np.unique(lab):
        ids = np.where(lab == c)[0]
        if len(ids) == 0:
            continue
        if DM is not None and len(ids) > 1:
            sub = DM[np.ix_(ids, ids)]
            mi = ids[int(np.argmin(sub.sum(1)))]
        else:
            mi = ids[0]
        col = cmap(c % 10)
        ax.plot(tr[:, mi, 0], tr[:, mi, 1], "-", color=col, lw=2.4, zorder=4,
                solid_capstyle="round")
        ax.scatter(tr[0, mi, 0], tr[0, mi, 1], s=26, marker="o", color=[col],
                   edgecolors="k", linewidths=0.6, zorder=5)
        ax.scatter(tr[-1, mi, 0], tr[-1, mi, 1], s=28, marker="D", color=[col],
                   edgecolors="k", linewidths=0.6, zorder=5)
    ax.set_xlabel("PC 1")
    extra = "" if d == REFDIM else f"   ({D[d]['overlap_ref']:.0%} vs d{REFDIM})"
    ax.set_title(f"d = {d}{extra}")


def fig_crossdim(panel=(3.9, 3.9), suptitle=None, title_top=0.975):
    ncols = len(AVAIL)
    fig = plt.figure(figsize=(panel[0] * ncols + RIGHT_W, panel[1]), dpi=150, layout="constrained")
    # `rect`, not `h_pad`: padding every axes collapses the nested right column into slivers
    fig.get_layout_engine().set(rect=(0, 0, 1, title_top))
    gs = fig.add_gridspec(1, ncols + 1, width_ratios=[1] * ncols + [RIGHT_W / panel[0]])
    axes = []
    for j, d in enumerate(AVAIL):
        ax = fig.add_subplot(gs[0, j], sharex=axes[0] if axes else None,
                             sharey=axes[0] if axes else None)
        draw_dim_panel(ax, d)
        axes.append(ax)
    axes[0].set_ylabel("PC 2")
    for ax in axes[1:]:
        ax.tick_params(labelleft=False)

    rgs = gs[0, ncols].subgridspec(3, 1, height_ratios=list(RIGHT_ROWS), hspace=RIGHT_HSPACE)
    ax_leg = fig.add_subplot(rgs[0]); ax_leg.axis("off")
    h = [Line2D([0], [0], color=cmap(c % 10), lw=2.4, label=f"cluster {c}")
         for c in np.unique(D[REFDIM]["clusters_m"])]
    h += [Line2D([0], [0], marker="o", ls="", mfc="0.5", mec="k", label="start", ms=6),
          Line2D([0], [0], marker="D", ls="", mfc="0.5", mec="k", label="end", ms=6)]
    leg = ax_leg.legend(handles=h, ncol=2, loc="upper left", bbox_to_anchor=(0.0, 1.0),
                        frameon=False, title=f"fate (matched to d={REFDIM})", handletextpad=0.5,
                        columnspacing=0.9, labelspacing=0.4, borderaxespad=0.0)
    leg._legend_box.align = "left"
    leg.set_in_layout(False)

    keys = list(pairs)
    A.metric_barh(fig.add_subplot(rgs[1]), keys, [pairs[k]["path_dev_pct"] for k in keys],
                  "trajectory deviation (%)", fs=FS, fmt="{:.1f}")
    A.metric_barh(fig.add_subplot(rgs[2]), keys, [pairs[k]["cluster_agree"] for k in keys],
                  "cluster agreement (%)", fs=FS, fmt="{:.0f}", higher_better=True)

    # descriptive title -- the stability claim is made in the text
    fig.suptitle(suptitle or (f"{DATASET}: trajectory comparison at d = "
                              f"{', '.join(str(d) for d in AVAIL)}"))
    return fig


fig_crossdim()
plt.show()